In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

seed = 1234
np.random.seed(seed)  


In [2]:
data = pd.read_csv('diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv')
features_to_drop = ['weight', 'payer_code', 'medical_specialty']
data = data.drop(features_to_drop, axis=1)
# data['medical_specialty'] = data['medical_specialty'].replace('?', 'Missing')
data = data.replace('?', np.nan)

#dropping rows with missing race for now
data = data.dropna(subset='race')
data['diag_1'] = pd.to_numeric(data['diag_1'], errors='coerce')

data['diag_1'] = data['diag_1'].fillna('Missing')

data['diag_2'] = pd.to_numeric(data['diag_2'], errors='coerce')

data['diag_2'] = data['diag_2'].fillna('Missing')

data['diag_3'] = pd.to_numeric(data['diag_3'], errors='coerce')

data['diag_3'] = data['diag_3'].fillna('Missing')
data.drop_duplicates(['patient_nbr'], keep = 'first', inplace = True)

print(len(data))

intervalDict = {'[0-10)' : 5,
'[10-20)' : 15,
'[20-30)' : 25, 
'[30-40)' : 35, 
'[40-50)' : 45, 
'[50-60)' : 55,
'[60-70)' : 65, 
'[70-80)' : 75,
'[80-90)' : 85,
'[90-100)' : 95}

data['age'] = data['age'].apply(lambda x : intervalDict[x])

# 1 = Emergency/Urgent, 5 = Not Available/NULL/Not Mapped, 3 = Elective, 4 = Newborn, 7 = Trauma Center
data['admission_type_id'] = data['admission_type_id'].apply(lambda x : 1 if int(x) in [1, 2] 
                                                            else (5 if int(x) in [5, 6, 8] 
                                                                  else int(x) ))
# 1 = Referral, 2 = Transfer, 3 = Nan, 4 = Delivery, etc. 
data['admission_source_id'] = data['admission_source_id'].apply(lambda x : 1 if int(x) in [2, 3] 
                                                                else (2 if int(x) in [4, 5, 6, 10, 22, 25] 
                                                                      else (3 if int(x) in [9, 15, 17, 20, 21] 
                                                                            else (4 if int(x) in [11, 13, 14] 
                                                                                  else int(x)))))



data.head()
# print(len(data))

69668


In [5]:
# split test/train

# separate features and target
X = data.drop("readmitted", axis=1)
y = data["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

print(X.shape)
print(X.columns)
# verify the shape
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(69668, 46)
Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed'],
      dtype='object')
(55734, 46)
(13934, 46)
(55734,)
(13934,)
